## pySpark DataFrame API: Column

### References:
1. pyspark Column: https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.Column.html

a Column is a fundamental programming construct representing a column in a DataFrame or Dataset. It acts as a Catalyst Expression that produces a value for each row in the dataset.

Core Functionality

- Column Referencing: You can reference columns in multiple ways, such as using the
  - pyspark.sql.functions.col function
  - dot notation (df.colName)
  - bracket notation (df["colName"]).
- Transformations: Columns are used to perform operations like
  - arithmetic
  - string manipulation
  - logical comparisons
- Immutability: Because Spark DataFrames are immutable, you cannot update a column "in place"; instead, you use methods like ***withColumn()*** to create a new DataFrame with the modified column. 

Common Column Operations

- Casting Types: Use the cast() method to change a column's DataType (e.g., from String to Integer).
- Conditional Logic: The when() and otherwise() methods allow for "if-then-else" logic within a column expression.
- String Filtering: Methods for text matching.
  - like()
  - contains()
  - startswith()
- Complex Data Types: For columns containing arrays or maps, the explode() function can be used to flatten them into multiple rows. 

Managing Columns in a DataFrame
- Listing Columns: Use df.columns to get a list of all column names in their defined order.
- Adding/Replacing: The withColumns() method allows adding or replacing multiple columns simultaneously using a dictionary mapping.
- Dropping Duplicates: The dropDuplicates() method can remove rows based on specific column values rather than the entire row. 


## pySpark DataFrame API: withColumn

### References

1. pyspark Column: https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.Column.html
2. pyspark Dataframe API: https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.withColumn.html


#### Introduction

The withColumn() function in PySpark is a transformation used to add a new column or replace an existing one by returning a new DataFrame. 

Core Functionality
- Add New Column: Creates a column if the name does not already exist.
- Replace Existing Column: Overwrites a column if the name matches an existing one.
- Immutability: Returns a new DataFrame; it does not modify the original in place. 

Common Use Cases
- Adding Constants: Use the lit() function from pyspark.sql.functions to add a fixed value.
- Type Casting: Change a column's data type using the cast() method within the function.
- Calculated Columns: Create new values based on arithmetic or string operations from other columns.
- Conditional Logic: Use when() and otherwise() to create columns based on specific conditions (similar to SQL CASE WHEN).  

### Setup

Setup sample DataFrame

In [2]:
from pyspark.sql import functions as F
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("DeptExamples").getOrCreate()

# Employee data: (emp_id, name, age, salary, dept_id)
emp_data = [(1, "Alice", 30, 70000, 10), 
            (2, "Bob", 40, 50000, 20),
            (3, "Charlie", 50, 80000, 10), 
            (4, "David", 55, 60000, 30),
            (5, "Eve", 56, 75000, 20),
            (6, "Frank", 57, 90000, 10)]

emp_df = spark.createDataFrame(emp_data, ["emp_id", "emp_name", "age", "salary", "dept_id"])
emp_df.show()

+------+--------+---+------+-------+
|emp_id|emp_name|age|salary|dept_id|
+------+--------+---+------+-------+
|     1|   Alice| 30| 70000|     10|
|     2|     Bob| 40| 50000|     20|
|     3| Charlie| 50| 80000|     10|
|     4|   David| 55| 60000|     30|
|     5|     Eve| 56| 75000|     20|
|     6|   Frank| 57| 90000|     10|
+------+--------+---+------+-------+



### Sample of using withColumn function to
1. Add a new column with constant value
2. Cast date type
3. Add new column using condition: when().otherwise()

In [14]:
from pyspark.sql.functions import col, lit, when

# 1. Add a new column with a constant value
emp_df = emp_df.withColumn("salary_percent_increase", lit(1))

# 2. Modify an existing column (e.g., cast to Integer)
emp_df = emp_df.withColumn("age", col("age").cast("integer"))

# 3. Create a column based on conditions
emp_df = emp_df.withColumn("status", when(col("age") <= 40, "Young").otherwise("Wiser"))
emp_df.show()


+------+--------+---+------+-------+-----------------------+------+
|emp_id|emp_name|age|salary|dept_id|salary_percent_increase|status|
+------+--------+---+------+-------+-----------------------+------+
|     1|   Alice| 30| 70000|     10|                      1| Young|
|     2|     Bob| 40| 50000|     20|                      1| Young|
|     3| Charlie| 50| 80000|     10|                      1| Wiser|
|     4|   David| 55| 60000|     30|                      1| Wiser|
|     5|     Eve| 56| 75000|     20|                      1| Wiser|
|     6|   Frank| 57| 90000|     10|                      1| Wiser|
+------+--------+---+------+-------+-----------------------+------+



### DF filter using Column function
1. Comparison opeations: <, >, etc...
2. contains, like(), startswith functions

In [8]:
# Using the DF dot notation
print("DF.filter: age> 50 and age<57 - Using DF dot notation")
emp_df.filter((emp_df.age > 50) & (emp_df.age < 57)).show()

# Using the col function
from pyspark.sql.functions import col

print("DF.filter: age> 50 and age<57 - Using funcion col")
emp_df.filter((col("age") > 50) & (col("age") < 57)).show()

print("DF.filter: age between 50 and 77 - inclusive")
emp_df.filter(col("age").between(50, 57)).show()

print("DF.filter: name contains e")
emp_df.filter(col("emp_name").contains("e")).show()

print("DF.filter: name like %v%")
emp_df.filter(col("emp_name").like("%v%")).show()

print("DF.filter: name startswith A")
emp_df.filter(col("emp_name").startswith("A")).show()

print("DF.filter: name in [Alice, Eve]")
emp_df.filter(col("emp_name").isin(["Alice","Eve"])).show()

DF.filter: age> 50 and age<57 - Using DF dot notation
+------+--------+---+------+-------+
|emp_id|emp_name|age|salary|dept_id|
+------+--------+---+------+-------+
|     4|   David| 55| 60000|     30|
|     5|     Eve| 56| 75000|     20|
+------+--------+---+------+-------+

DF.filter: age> 50 and age<57 - Using funcion col
+------+--------+---+------+-------+
|emp_id|emp_name|age|salary|dept_id|
+------+--------+---+------+-------+
|     4|   David| 55| 60000|     30|
|     5|     Eve| 56| 75000|     20|
+------+--------+---+------+-------+

DF.filter: age between 50 and 77 - inclusive
+------+--------+---+------+-------+
|emp_id|emp_name|age|salary|dept_id|
+------+--------+---+------+-------+
|     3| Charlie| 50| 80000|     10|
|     4|   David| 55| 60000|     30|
|     5|     Eve| 56| 75000|     20|
|     6|   Frank| 57| 90000|     10|
+------+--------+---+------+-------+

DF.filter: name contains e
+------+--------+---+------+-------+
|emp_id|emp_name|age|salary|dept_id|
+------